# RECRAG 생성 모듈 — QLoRA 파인튜닝 (Colab T4, 원클릭)

위에서부터 셀을 **순서대로 실행**하면 다음이 모두 수행됩니다.

1. GPU 확인 → 2. repo clone + 의존성 설치 → 3. (선택) 학습셋 재생성 →
4. **QLoRA 학습** → 5. base vs 파인튜닝 **비교 추론** → 6. (선택) 어댑터를 Google Drive에 저장

런타임: **수정 → 런타임 유형 변경 → T4 GPU** 로 먼저 설정하세요.

> repo에 이미 합성 학습셋(`finetune/data/sft_train.jsonl`, `sft_val.jsonl`)이 포함돼 있어,
> **3번을 건너뛰고 바로 4번(학습)** 부터 돌려도 됩니다. (3번 재생성에는 `UPSTAGE_API_KEY` 필요)

## 0. GPU 확인 (T4 인지 체크)

In [ ]:
!nvidia-smi

## 1. repo clone + 의존성 설치

In [ ]:
REPO_URL = "https://github.com/haneul-dev/recrag-generation-module.git"
BRANCH   = "feat/qlora-finetune"   # 머지 후엔 "main" 으로 바꾸면 됩니다

%cd /content
!rm -rf recrag-generation-module
!git clone -b $BRANCH --single-branch $REPO_URL
%cd recrag-generation-module
!pip -q install -r requirements.txt
!pip -q install -r finetune/requirements_finetune.txt

## 2. (선택) teacher API 키 — 학습셋을 재생성할 때만 필요
repo에 학습셋이 이미 있으므로 **재생성이 필요 없으면 2·3번을 건너뛰세요.**

In [ ]:
import os, getpass
os.environ["UPSTAGE_API_KEY"] = getpass.getpass("UPSTAGE_API_KEY (없으면 빈칸): ")

## 3. (선택) 합성 학습셋 재생성

In [ ]:
# repo의 기존 학습셋을 새로 만들고 싶을 때만 실행 (UPSTAGE_API_KEY 필요)
!python finetune/build_sft_dataset.py --config finetune/finetune_config.yaml
!wc -l finetune/data/sft_train.jsonl finetune/data/sft_val.jsonl

## 4. QLoRA 학습 (T4에서 수십 분 예상)
결과 어댑터: `finetune/outputs/qwen2.5-7b-recrag-qlora/`

In [ ]:
!python finetune/train_qlora.py --config finetune/finetune_config.yaml

## 5. base vs 파인튜닝 비교 추론
동일한 평가 쿼리에 대해 **원본(base)** 과 **파인튜닝(base+어댑터)** 출력을 나란히 비교합니다.
(T4 메모리 절약을 위해 순차 로드)

In [ ]:
import sys, gc, yaml, torch
sys.path.insert(0, "src")
from data_loader import load_eval_set, filter_chunks
from prompt_builder import build_messages
from llm_runner import LLMRunner

ADAPTER = "finetune/outputs/qwen2.5-7b-recrag-qlora"
N = 3  # 비교할 쿼리 수
rows = load_eval_set("data/eval_set.manual_top5.jsonl")[:N]

def generate_all(adapter_path):
    cfg = yaml.safe_load(open("generation_experiment_config.yaml"))
    cfg["model"]["quantization"] = "4bit"
    if adapter_path:
        cfg["model"]["adapter_path"] = adapter_path
    runner = LLMRunner(cfg).load()
    outs = []
    for r in rows:
        f = filter_chunks(r["retrieved_chunks"])
        msgs = build_messages(r["query"], f["used_chunks"])
        outs.append(runner.generate(msgs)["output_text"])
    del runner; gc.collect(); torch.cuda.empty_cache()
    return outs

print("base 추론 중...");      base_out = generate_all(None)
print("파인튜닝 추론 중...");  ft_out   = generate_all(ADAPTER)

for i, r in enumerate(rows):
    print("=" * 90)
    print("Q:", r["query"])
    print("\n[BASE]\n", base_out[i])
    print("\n[FINETUNED]\n", ft_out[i])

## 6. (선택) 어댑터를 Google Drive에 저장
Colab 런타임이 끊겨도 어댑터를 보존합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/RECRAG/qlora_adapter"
!cp -r finetune/outputs/qwen2.5-7b-recrag-qlora/* "/content/drive/MyDrive/RECRAG/qlora_adapter/"
print("저장 완료: /content/drive/MyDrive/RECRAG/qlora_adapter")